In [ ]:
import pandas as pd

: 

In [ ]:
train = pd.read_csv('data/train.csv')
print(train.head(5))

In [ ]:
train.columns

In [ ]:
test = pd.read_csv('data/test.csv')
print(test.head(5))

#### Punto 2.
El dataset de Test no tiene la variable "survived" porque esta variable es de prediccion y no queremos que nuestro modelo aprenda de memoria al predecir

In [ ]:
test.columns

In [ ]:
data_dict = {
    "Variable": ["PassengerId", "Survived", "Pclass", "Name", "Sex", "Age", "SibSp", "Parch", "Ticket", "Fare", "Cabin", "Embarked"],
    "Descripción": [
        "Identificador único",
        "0 = No sobrevivió, 1 = Sobrevivió",
        "1 = 1ª Clase, 2 = 2ª Clase, 3 = 3ª Clase",
        "Nombre y título",
        "Género",
        "Edad en años",
        "Nº de hermanos/cónyuges a bordo",
        "Nº de padres/hijos a bordo",
        "Número del ticket",
        "Tarifa pagada",
        "Número de camarote",
        "Puerto de embarque (C, Q, S)"
    ]
}


df_diccionario = pd.DataFrame(data_dict)


print("Diccionario de datos del Titanic:")
display(df_diccionario)

#### Punto 3
Comparacion de variables:

* Se puede ver que ambos dataframes, el de test y el de train, son bastanantes parecidos:
* En la edad el promedio es casi igual en ambos dataframes, igual en las tarifas
* Tenemos datos incompletos en ambos dataframes, por ejemplo en edad nos faltan datos en ambos
* En test no existe la columna survive

In [ ]:
# Estadísticas del set de entrenamiento
print("Estadisticas descriptivas de TRAIN:")
display(train.describe())

print("\n--------------------------------------------------\n")

# Estadísticas del set de prueba
print("Estadisticas descriptivas de TEST:")
display(test.describe())

#### Punto 4
SibSp: Es el número de hermanos o cónyuges

Parch: Es el numero de padres e hijos

¿Que opinamos de esta nueva variable?

Respuesta: Es mejor crear esta nueva variable ya que podemos simplificar las variables, asi podemos saber simplemente si viaja con alguien o viaja solo. Con una sola variable, al momento de predecir podria ser mejor tener menos variables

In [ ]:
train['Familiares']= train['SibSp'] + train['Parch']
test['Familiares']= train['SibSp'] + train['Parch']
print(train[['SibSp', 'Parch', 'Familiares']].sample(10))

#### punto 5.
Se dejaran sus valores NaN, no se debe borrar ya que permite identificar las filas de prueba y representa la variable que queremos predecir mas adelante

In [ ]:
train['source'] = 'train'
test['source'] = 'test'

# pd.concat' las pega verticalmente
# ignore_index=True' rearma la numeración de las filas desde 0

datos_totales=pd.concat([train, test], ignore_index=True)
print(datos_totales)

#### punto 6
Despues de unir los dos conjuntos, la columna Survived se mantiene en el DataFrame final, para las filas que vienen de 'Test' Pandas les asignara automaticamente valores vacios, y eso esta bien porque no sabemos cuales sobrevivieron, entonces se dejara tal cual

In [ ]:
estadisticas_comparadas = datos_totales.groupby('source').describe()
print(datos_totales.groupby('source')['Age'].describe())
print(datos_totales.groupby('source')['Fare'].describe())

#### Punto 7.
Tras analizar el conjunto de datos, se identificó que las variables con datos faltantes son: Cabin (77.46%)
Survived (31.93% —correspondiente exactamente al test que no incluía esta variable—), Age (20.09%), Embarked (0.15%) y Fare (0.08%). La columna con mayor ausencia de información es Cabin (Cabina), la cual representa el número de camarote asignado al pasajero. En este contexto, el alto porcentaje de datos faltantes significa que no existió un registro exhaustivo de los camarotes para todas las clases, siendo una información casi exclusiva de los pasajeros de Primera Clase. Para el resto de los tripulantes y pasajeros (especialmente Tercera Clase), esta información es desconocida o se perdió durante el naufragio.

In [ ]:
# Muestra la cantidad exacta de valores nulos por cada columna
print(datos_totales.isnull().sum())
print("----------------------")
# Muestra el porcentaje de datos faltantes
print((datos_totales.isnull().sum() / len(datos_totales)) * 100)


#### Punto 8

A : Edad Promedio

In [ ]:
print(train['Age'].mean())


B : Sobrevivieron y murieron

In [ ]:
print(train['Survived'].value_counts())

C :  Tarifa promedio primera clase

In [ ]:
print(train[train['Pclass'] == 1]['Fare'].mean())

D : Pasajeros con familiar a bordo

In [ ]:
con_familiar = train[train['Familiares'] > 0]
print(f"Pasajeros con familiar: {len(con_familiar)}")

E : Edad más joven y más vieja

In [ ]:
print(f"Edad más joven: {train['Age'].min()} años")
print(f"Edad más vieja: {train['Age'].max()} años")

F : Pasajeros por puerto

In [ ]:
print(train['Embarked'].value_counts())

G : Solos vs con familiares

In [ ]:
solos = train[train['Familiares'] == 0]
con_familia = train[train['Familiares'] > 0]
print(f"Viajaron solos: {len(solos)}")
print(f"Viajaron con familiares: {len(con_familia)}")

#### Punto 9

A : Sobrevivieron mas mujeres que hombres, tanto en cantidad como proporcionalmente, porque en el Titanic se priorizó evacuar primero a mujeres y niños

In [ ]:
supervivencia_sexo = train.groupby('Sex')['Survived'].agg(['sum', 'mean'])
supervivencia_sexo.columns = ['Total_sobrevivientes', 'Proporcion']
print(supervivencia_sexo)

B : Creo que sobrevivieron más niños proporcionalmente por la misma razón, se les dio prioridad en los botes salvavidas.

In [ ]:
tabla_9b = train.groupby('Sex')['Survived'].agg(['sum', 'mean'])
tabla_9b.columns = ['Total', 'Proporcion']
print(tabla_9b)

C : Creo que sí sobrevivieron más, los niños por prioridad de evacuación y los mayores posiblemente viajaban en clases más altas

In [ ]:
tabla_9c = train.groupby('Pclass')['Survived'].agg(['sum', 'mean'])
tabla_9c.columns = ['Total', 'Proporcion']
print(tabla_9c)

D : Creo que en cantidad sí salieron más de Southampton porque era el puerto principal, pero proporcionalmente no necesariamente

In [ ]:
supervivencia_puerto = train.groupby('Embarked')['Survived'].agg(['sum', 'mean'])
supervivencia_puerto.columns = ['Total', 'Proporcion']
print(supervivencia_puerto)

E : Creo que los de primera clase sobrevivieron más proporcionalmente porque tenían mejor acceso a los botes al estar en cubiertas superiores 

In [ ]:
supervivencia_clase = train.groupby('Pclass')['Survived'].agg(['sum', 'mean'])
supervivencia_clase.columns = ['Total', 'Proporcion']
print(supervivencia_clase)

#### Punto 10

Grupos familiares y distribución de cabinas

La mayoría de pasajeros sin familiares y con familias pequeñas 
no tenían cabina asignada (NaN). Las familias grandes tampoco 
tenían muchas cabinas registradas.

In [ ]:
# Crear los grupos familiares
def grupo_familiar(f):
    if f == 0:
        return 'Grupo 1: Sin familiares'
    elif f <= 3:
        return 'Grupo 2: Familia pequeña'
    else:
        return 'Grupo 3: Familia grande'

train['grupo_familiar'] = train['Familiares'].apply(grupo_familiar)

# Distribución de cabinas por grupo
print(train.groupby('grupo_familiar')['Cabin'].value_counts())

#### Punto 11

Proporción de supervivencia por sexo, grupo etario y cabina

Los datos muestran que las mujeres sobrevivieron más en todos los 
grupos etarios. Los pasajeros con cabina asignada tuvieron mayor 
tasa de supervivencia, lo que sugiere que viajar en clases superiores 
aumentaba las probabilidades de sobrevivir.

In [ ]:
# Crear grupo etario
def grupo_etario(age):
    if age < 10:
        return 'Niño'
    elif age < 50:
        return 'Adulto'
    else:
        return 'Mayor'

train['grupo_etario'] = train['Age'].apply(grupo_etario)

# Crear columna tiene_cabina
train['tiene_cabina'] = train['Cabin'].notna()

# Tabla de proporción
tabla = train.groupby(['grupo_etario', 'Sex', 'tiene_cabina'])['Survived'].mean()
print(tabla)

#### Punto 12

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


# 1
plt.figure(figsize=(10, 5))
sns.histplot(data=datos_totales, x="Age", hue="source", kde=True, bins=30, palette="Set2")
plt.title("Distribucion de edades. Train vs Test en el Titanic")
plt.xlabel("Edad")
plt.ylabel("Cantidad de pasajeros")
plt.show()

# 2
plt.figure(figsize=(10, 5))
sns.barplot(data=train, x="Pclass", y="Survived", hue="Sex", palette="pastel")
plt.title("Tasa de supervivencia por clase y sexo")
plt.xlabel("Clase del tiquete")
plt.ylabel("Tasa de Supervivencia")
plt.show()

: 

### Analisis de graficos

1. Grafico de distribucion de edades:
Al comparar la distribución de las edades en ambos conjuntos de datos (train y test), podemos observar que la distribucion de las dos son muy parecidas, las dos concentran sus datos en el rango de 20 y 30 años esto es muy importante al momento de interpretar por que nos dice que los dos grupos de datos son representativos entre si, el conjunto test es altamente representativo de train, asi que se que al de hacer un modelo predictivo al tener poblaciones iguales va a ser justo  entonces va a ser mas preciso.

2. Grafico de tasa de supervivencia por clase y Sexo:
Este grafico nos muestra la clara desigualdad que hay entre clases y sexo en el barco. Se puede observar que las personas con mayor tasa de supervivencia son las mujeres que viajaban en primera y segunda clase. Mientras se puede ver que en tercera clase y mas por parte de hombres baja extremadamente esta tasa de supervivencia. Esto nos dice que el protocolo de seguridad del barco priorizo claramente a las personas que pagaron mas y si eran mujeres o niños.

#### Punto 13

¿QUIEN TENIA MAS PROBABILIDADES DE SOBREVIVIR?

Para determinar qué perfil tenía mayores probabilidades de sobrevivir, analizamos los datos bajo el concepto de probabilidad condicional. Es decir, calculamos la probabilidad de supervivencia  dado un conjunto de condiciones especificas (genero y clase).Matemáticamente, lo que hace nuestro modelo al agrupar los datos en Pandas es calcular la siguiente probabilidad :

- P(S(G n C))  

- S=El pasajero sobrevive
- C= La clase del tiquete (C1, C2, C3).
- G= El genero (F:femenino,M:masculino)

Al evaluar el espacio muestral completo, encontramos que la intersección con la probabilidad de exito mas alta se da en el evento de ser mujer en primera clase:

- P(S (F n C1))  = 0.96

Ya con esto queda demostrado que dado que un pasajero fuera mujer y estubiera en primera clase su probabilidad de sobrevivir era del 96%.


Basándonos en este analisis, el perfil con la mayor probabilidad matemática de sobrevivir al hundimiento del Titanic era una mujer, que viajara en Primera Clase y que tuviera una cabina asignada.

Los datos demuestran que las mujeres de Primera Clase superaron el 96% de tasa de supervivencia. Por el otro lado, los pasajeros con la probabilidad más baja fueron los hombres adultos que viajaban en tercera clase . El estatus del tiquete y el género fueron los factores que hicieron que la gente obtuviera mas probabilidades de sobrevivir o no.